#Phase 1: Inspect and Clean
## Step 2: Load the combined dataset
Load all 212 anonymized patients from one file and fix the first column name

In [17]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "phase1_data" else Path.cwd()
path = ROOT / "data" / "raw" / "all_patients.csv"

df = pd.read_csv(path, dtype=str, encoding="utf-8-sig", engine="c", low_memory=False)
df = df.rename(columns={df.columns[0]: "patient"})

funnel = [("Loaded", len(df), df["patient"].nunique())]
print("Rows:", df.shape[0], "| Columns:", df.shape[1], "| Patients:", df["patient"].nunique())

Rows: 117474 | Columns: 115 | Patients: 212


##Step 2: Columns Overview 
Chcek each column's missing %, unique values, and average text length to find useful and useless columns.

In [18]:
overview = pd.DataFrame({
    "missing_%": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(),
    "avg_length": df.apply(lambda s: s.dropna().str.len().mean()).round(1),
}).sort_values("missing_%")

print("Columns 100% empty:", (overview["missing_%"] == 100).sum())
print("Columns with 1 value only:", (overview["unique"] <= 1).sum())
overview.head(30)

Columns 100% empty: 15
Columns with 1 value only: 10


,missing_%,unique,avg_length
patient,0.0,212,4.0
Field2,23.6,81854,22.7
Field3,23.6,1,3.0
Field48,23.6,13142,12.9
Field4,31.2,1,12.0
Field1_Text,40.7,213,4.1
Field87,40.7,211,4.1
Field47,68.9,32650,171.1
Field34,76.4,25216,22.5
Field41,76.4,25213,22.5


## Step 3: Posts per patient
Count how many rows each patient has, to spot patients with too little data.

In [19]:
posts = df["patient"].value_counts()
print("Patients:", posts.size)
print("Rows per patient → min:", posts.min(), "| median:", int(posts.median()), "| max:", posts.max())
print("Patients with fewer than 5 rows:", (posts < 5).sum())

Patients: 212
Rows per patient → min: 19 | median: 583 | max: 2181
Patients with fewer than 5 rows: 0


## Step 4: Drop useless columns
Remove columns that are completely empty or have the same value in every row, since they carry no information.

In [20]:
before = df.shape[1]
empty = [c for c in df.columns if df[c].isna().all()]
constant = [c for c in df.columns if c != "patient" and df[c].nunique() <= 1]
df = df.drop(columns=list(set(empty + constant)))

print("Columns before:", before, "| removed:", before - df.shape[1], "| now:", df.shape[1])

Columns before: 115 | removed: 10 | now: 105


## Step 5: Keep only the patient's own posts
Keep rows where the author is the patient, and drop posts by other people ("OTHER") or with no author, since only the patient's own words show their language.

In [21]:
before = len(df)
df = df[df["Field1_Text"] == df["patient"]]
funnel.append(("Patient's own posts only", len(df), df["patient"].nunique()))

print("Rows before:", before, "| removed:", before - len(df), "| now:", len(df))
print("Patients still present:", df["patient"].nunique())

Rows before: 117474 | removed: 54443 | now: 63031
Patients still present: 212


## Step 6: Find the text columns
Rank the remaining columns by how many rows they fill and their average length, to find where the post text is stored.

In [22]:
meta = {"patient", "Field1_Text", "Field2", "Field87", "Time", "Time1"}
cands = [c for c in df.columns if c not in meta]

text_rank = pd.DataFrame({
    "filled_rows": df[cands].notna().sum(),
    "avg_length": df[cands].apply(lambda s: s.dropna().str.len().mean()).round(1),
})
text_rank = text_rank[text_rank["filled_rows"] > 0].sort_values("filled_rows", ascending=False)
text_rank.head(25)

,filled_rows,avg_length
Field48,63027,4.2
Field71,25308,21.3
Field35,25307,3.0
Field34,25305,22.4
Field21_Text,23891,15.3
Field61,23891,19.8
Field47,23648,157.8
Field69,12978,4.6
Field79,11737,5.1
Field36,11064,146.2


## Step 7: Combine the text columns into one
Join all long-text columns into a single `text` column per post, removing repeated copies of the same text.